# Exploratory Data Analysis

FMA-small dataset statistics and verified splits. Populated in Phase 1.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import matplotlib.pyplot as plt

from src.datasets import compute_dataset_statistics, load_fma_metadata, load_fma_splits
from src.utils import load_config

config = load_config(Path.cwd().parent / "config.yaml")
subset = config["dataset"]["name"].replace("fma_", "")
tracks = load_fma_metadata(Path.cwd().parent / "data/raw/fma_metadata", subset=subset)
splits = load_fma_splits(tracks)
stats = compute_dataset_statistics(tracks)
stats

## Split sizes and genre distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(splits.keys(), [len(v) for v in splits.values()])
axes[0].set_title(f"FMA-{subset} split sizes (official, artist-disjoint)")
axes[0].set_ylabel("# tracks")

genre_counts = tracks["genre_top"].value_counts()
axes[1].barh(genre_counts.index[::-1], genre_counts.values[::-1])
axes[1].set_title("Top-level genre distribution")

plt.tight_layout()
plt.savefig(Path.cwd().parent / "results/plots/fma_split_and_genre_distribution.png", dpi=150)
plt.show()

## Text-source coverage for Task 1 (BERT tag classifier)

Only ~18% of tracks have non-empty `track_tags`, but ~70% have an `artist_bio`.
Note: artist bios can literally contain the genre word (e.g. a Hip-Hop artist's
bio mentioning "Hip-Hop"); the raw bio is used as-is for Task 1 input text.


In [ ]:
from collections import Counter

tag_counter = Counter()
for tags in tracks["track_tags"]:
    tag_counter.update(tags)

print(f"tracks with non-empty track_tags: {stats['num_tracks_with_track_tags']} / {stats['num_tracks']}")
print(f"tracks with artist_bio: {stats['num_tracks_with_artist_bio']} / {stats['num_tracks']}")
print(f"unique track tags: {len(tag_counter)}")

top_tags = tag_counter.most_common(30)
plt.figure(figsize=(8, 8))
plt.barh([t for t, _ in top_tags][::-1], [c for _, c in top_tags][::-1])
plt.title("Top-30 track tags (candidate multi-label vocabulary for Task 1)")
plt.tight_layout()
plt.savefig(Path.cwd().parent / "results/plots/fma_top_tags.png", dpi=150)
plt.show()

## Phase 2: Audio preprocessing inspection

Load, resample (22050 Hz), peak-normalize, segment into fixed 5s windows, and
extract chroma+MFCC node features for one track per genre. Verifies no
NaN/Inf in features (project validation check #6).

In [ ]:
import librosa.display

from src.audio_features import extract_segment_features, load_audio, segment_audio

sr = config["dataset"]["sample_rate"]
genres_to_inspect = ["Rock", "Hip-Hop", "Classical", "Electronic"]
picked = [tracks[tracks["genre_top"] == g].sample(1, random_state=7).iloc[0] for g in genres_to_inspect]

fig, axes = plt.subplots(len(picked), 3, figsize=(15, 4 * len(picked)))
for i, row in enumerate(picked):
    tid = int(row["track_id"])
    tid_str = f"{tid:06d}"
    path = Path.cwd().parent / "data/raw/fma_medium" / tid_str[:3] / f"{tid_str}.mp3"
    y = load_audio(str(path), sample_rate=sr)
    segments = segment_audio(y, sr, config["audio"]["segment_seconds"])
    feats = [
        extract_segment_features(s, sr, config["audio"]["n_mfcc"], config["audio"]["use_chroma"])
        for s in segments
    ]
    print(
        f"track {tid} genre={row['genre_top']} n_segments={len(segments)}",
        f"nan={any(not __import__('numpy').isfinite(f).all() for f in feats)}",
    )

    axes[i, 0].plot(librosa.times_like(y, sr=sr), y, linewidth=0.3)
    axes[i, 0].set_title(f"{row['genre_top']} (id={tid}) waveform")
    mel_db = librosa.power_to_db(librosa.feature.melspectrogram(y=y, sr=sr, n_mels=config["audio"]["n_mels"]))
    librosa.display.specshow(mel_db, sr=sr, ax=axes[i, 1], x_axis="time", y_axis="mel")
    axes[i, 1].set_title("log-mel spectrogram")
    chroma = librosa.feature.chroma_stft(y=y, sr=sr)
    librosa.display.specshow(chroma, sr=sr, ax=axes[i, 2], x_axis="time", y_axis="chroma")
    axes[i, 2].set_title("chroma")

plt.tight_layout()
plt.savefig(Path.cwd().parent / "results/plots/phase2_track_inspection.png", dpi=130)
plt.show()